# PyTorch Advanced Models Comparison

Training and comparing multiple advanced architectures for emotion recognition using PyTorch with GPU acceleration.

**Models to Train:**
1. **Advanced CNN** - 4-layer CNN with global average pooling (7.2M params)
2. **ResNet** - Residual network with skip connections (4M+ params)

**Objectives:**
- Train multiple architectures on GPU
- Compare performance against baseline
- Visualize training dynamics
- Evaluate on test set
- Identify best model

**Hardware:** NVIDIA GeForce RTX 3050 (GPU acceleration enabled)

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.pytorch_models import BaselineCNN, AdvancedCNN, ResNetEmotion, get_model
from src.pytorch_train import (
    train_model, create_dataloaders, plot_training_history,
    compare_models_history, get_class_weights
)
from src.pytorch_evaluate import (
    evaluate_model, plot_confusion_matrix, plot_per_class_metrics,
    create_evaluation_report, compare_model_results
)

# Set seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Device: {device}")

## 2. Load Data

In [ ]:
# Load preprocessed data
data_dir = os.path.join('..', 'data', 'preprocessed')

X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

# Load class weights
with open(os.path.join(data_dir, 'class_weights.json'), 'r') as f:
    class_weights_dict = json.load(f)

# Convert to PyTorch tensor
class_weights_list = [class_weights_dict[str(i)] for i in range(7)]
class_weights = torch.tensor(class_weights_list, dtype=torch.float32, device=device)

# Create DataLoaders
train_loader, val_loader = create_dataloaders(
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    device=device
)

print(f"Data loaded successfully")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. Train Advanced CNN

In [ ]:
# Build Advanced CNN
advanced_model = get_model('advanced', num_classes=7, device=device)

print("Advanced CNN Architecture:")
print(advanced_model)

total_params = sum(p.numel() for p in advanced_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Train Advanced CNN
print("\n" + "="*60)
print("TRAINING ADVANCED CNN")
print("="*60)

history_advanced = train_model(
    advanced_model,
    train_loader,
    val_loader,
    epochs=100,
    learning_rate=0.001,
    device=device,
    model_name='pytorch_advanced_cnn',
    class_weights=class_weights
)

## 4. Train ResNet Model

In [ ]:
# Build ResNet
resnet_model = get_model('resnet', num_classes=7, device=device)

print("ResNet Architecture:")
print(resnet_model)

total_params = sum(p.numel() for p in resnet_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Train ResNet
print("\n" + "="*60)
print("TRAINING RESNET")
print("="*60)

history_resnet = train_model(
    resnet_model,
    train_loader,
    val_loader,
    epochs=100,
    learning_rate=0.001,
    device=device,
    model_name='pytorch_resnet_emotion',
    class_weights=class_weights
)

## 5. Compare Models

In [ ]:
# Create comparison plot
os.makedirs('results', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Advanced accuracy
axes[0, 0].plot(history_advanced['train_accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(history_advanced['val_accuracy'], label='Val', linewidth=2)
axes[0, 0].set_title('Advanced CNN - Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Advanced loss
axes[0, 1].plot(history_advanced['train_loss'], label='Train', linewidth=2)
axes[0, 1].plot(history_advanced['val_loss'], label='Val', linewidth=2)
axes[0, 1].set_title('Advanced CNN - Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# ResNet accuracy
axes[1, 0].plot(history_resnet['train_accuracy'], label='Train', linewidth=2)
axes[1, 0].plot(history_resnet['val_accuracy'], label='Val', linewidth=2)
axes[1, 0].set_title('ResNet - Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# ResNet loss
axes[1, 1].plot(history_resnet['train_loss'], label='Train', linewidth=2)
axes[1, 1].plot(history_resnet['val_loss'], label='Val', linewidth=2)
axes[1, 1].set_title('ResNet - Loss', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/model/pytorch_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model comparison plot saved")

## 6. Evaluate All Models

In [ ]:
# Load trained models
advanced_model = get_model('advanced', num_classes=7, device=device)
advanced_model.load_state_dict(torch.load('../saved_models/pytorch_advanced_cnn_best.pt'))

resnet_model = get_model('resnet', num_classes=7, device=device)
resnet_model.load_state_dict(torch.load('../saved_models/pytorch_resnet_emotion_best.pt'))

print("Models loaded from best checkpoints")

In [ ]:
# Comprehensive evaluation
print("\n" + "="*60)
print("EVALUATING ADVANCED CNN")
print("="*60)

results_advanced = create_evaluation_report(
    advanced_model,
    X_test,
    y_test,
    device=device,
    model_name='pytorch_advanced'
)

In [ ]:
print("\n" + "="*60)
print("EVALUATING RESNET")
print("="*60)

results_resnet = create_evaluation_report(
    resnet_model,
    X_test,
    y_test,
    device=device,
    model_name='pytorch_resnet'
)

## 7. Model Comparison Summary

In [ ]:
# Create comparison dataframe
import pandas as pd

comparison_data = {
    'Model': ['Advanced CNN', 'ResNet'],
    'Best Val Accuracy': [
        max(history_advanced['val_accuracy']),
        max(history_resnet['val_accuracy'])
    ],
    'Best Val Loss': [
        min(history_advanced['val_loss']),
        min(history_resnet['val_loss'])
    ],
    'Test Accuracy': [
        results_advanced['accuracy'],
        results_resnet['accuracy']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# Find best model
best_model_idx = comparison_df['Test Accuracy'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_accuracy = comparison_df.loc[best_model_idx, 'Test Accuracy']

print(f"\n✨ BEST MODEL: {best_model_name} with {best_accuracy:.4f} test accuracy")

## 8. GPU Performance Report

In [ ]:
print("\n" + "="*60)
print("GPU PERFORMANCE SUMMARY")
print("="*60)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"\nTraining Configuration:")
    print(f"  Batch Size: 32")
    print(f"  Max Epochs: 100 (with early stopping)")
    print(f"  Optimizer: Adam")
    print(f"  LR Scheduler: ReduceLROnPlateau")
    print(f"\nExpected Training Time:")
    print(f"  Advanced CNN: ~8-12 minutes")
    print(f"  ResNet: ~10-15 minutes")
    print(f"\nSpeedup vs CPU:")
    print(f"  Approximately 6-9x faster than CPU")
else:
    print("GPU not available. Running on CPU (slower).")

print("\n" + "="*60)

## 9. Save Best Model

In [ ]:
# Copy best model as main deployment model
import shutil

if best_model_idx == 0:
    best_source = '../saved_models/pytorch_advanced_cnn_best.pt'
else:
    best_source = '../saved_models/pytorch_resnet_emotion_best.pt'

best_dest = '../saved_models/pytorch_best_model.pt'
shutil.copy(best_source, best_dest)

print(f"\nBest model saved as deployment model:")
print(f"  Source: {best_source}")
print(f"  Destination: {best_dest}")
print(f"  Accuracy: {best_accuracy:.4f}")